Hi! Welcome to my Physics-Informed Sybolic Regression (PISR) Heat Diffusion Toy Model.

This model uses PySR and SymPy to predict the heat equation describing a data set. The goal is to show how a PISR pipeline might work when we want to extract an analytical equation from a dataset where we know the physical laws that equation must obey. In our case, we have a dataset that we know describes some sort of diffusion, but we want to make sure that whatever PySR comes up with must solve the heat diffusion equation, thereby following physical laws.

Let us first import some neccessary modules: numpy, pysr, and sympy

In [ ]:
!pip install pysr
import pysr # This installs the Julia backend required for PySRRegressor
pysr.install()

import sympy as sp
import numpy as np
from pysr import PySRRegressor

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 8.9 MB/s eta 0:00:00
[juliapkg] Found dependencies: /usr/local/lib/python3.13/dist-packages/juliapkg/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.13/dist-packages/juliacall/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.13/dist-packages/pysr/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] WARNING: You have Julia 1.12.6 installed but 1.10.3 - 1.11 is required.
[juliapkg]   It is recommended that you upgrade Julia or install JuliaUp.
[juliapkg] Querying Julia versions from https://julialang-s3.julialang.org/bin/versions.json
[juliapkg] WARNING: About to install Julia 1.11.9 to /root/.julia/environments/pyjuliapkg/pyjuliapkg/install.
[juliapkg]   If you use juliapkg in more than one environment, you are likely to


/usr/local/lib/python3.13/dist-packages/pysr/deprecated.py:10: FutureWarning: The `install` function has been removed. PySR now uses the `juliacall` package to install its dependencies automatically at import time. 
  warnings.warn(


Since this is a toy, we first pick a valid know u(x,t) that solves the heat equation, u_t = a*u_xx, where a is the thermal diffusivity constant.

In [ ]:
def actual_eqn(t,x,a):
  # a is the thermal diffusivity constant
  return np.exp(-a*t) * np.sin(x) # PLACE YOUR EQUATION HERE!

# Spacetime grid
t_vals = np.linspace(0,5,50)
x_vals = np.linspace(0, 2*np.pi, 100)
T_grid, X_grid = np.meshgrid(t_vals, x_vals)

U_grid = actual_eqn(T_grid, X_grid,0.1) # true values for grid

Now we can generate our "fake data". We will synthetically inject noise (Gaussian/Normal noise) into our U_grid, thus simulating measured values.

In [ ]:
noise_level = 0.05 # standard deviation
noise = np.random.normal(0, noise_level, U_grid.shape)
U_grid_noisy = U_grid + noise

Now, we have to flatten (make our values one row) our T_Grid, X_Grid, and U_grid_noisy. Doing so creates a sort of "spreadsheet", with our feautures as t_flat and x_flat, and our target as u_flat. We then column stack our features, which we call X, to later feed into PySR. For consistency, we call u_flat = Y.

In [ ]:
t_flat = T_grid.flatten()
x_flat = X_grid.flatten()

X = np.column_stack((t_flat, x_flat)) # [Time, Space]
Y = U_grid_noisy.flatten() # Temperature

Now we can start doing some symbolic regression! First we build the model:

`niterations` refers to the number of "generations" we want the model to run though.

`binary_operators` and unary_operators let us select the options the model can use in its equations.

`extra_sympy_mappings` allows us to translate Julia to Python. When Julia reads back "exp", it does so as a string. This lets us redefine that string as a sp object such that python can work with it.

`elementwise_loss` is the cost function we choose to evaluate the performance of an equation relative to the data

`maxsize` = Number of "nodes" (operators, variables, constants) allowed in an equation. Limits complexity so it doesn't give us massive unreadable polynomials.

In [ ]:
model = PySRRegressor(
    niterations=40,
    binary_operators=["+", "*", "-"],
    unary_operators=["exp", "sin"],
    extra_sympy_mappings={"exp": sp.exp, "sin": sp.sin},
    elementwise_loss="loss(prediction, target) = (prediction - target)^2",
    maxsize=15
)

# We set the variable names so that the generated SymPy equations use t and x
model.fit(X, Y, variable_names=['t', 'x'])

# PySR automatically saves the best candidate equations in a Pandas DataFrame
candidates = model.equations_

Compiling Julia backend...
INFO:pysr.sr:Compiling Julia backend...


Now that we have candidate equations from PySR, we can evaluate them against how well they fit the heat equation residual:

First we define our SymPy symbols. Don't forget to put in your thermal diffusion constant, a!

Then we create a loop to interatively go through and rate each equation. We store these ranked equations in a dictionary, sort it by best total loss score, and print the top 10 equations.



In [ ]:
t, x = sp.symbols('t x')
a = 0.1
weight = 10.0 # Our hyperparameter to balance data vs. physics

evaluated_equations = [] # list to store evaluated equations

for i, row in candidates.iterrows():

  # get SymPy expression of equation
  proposed_equation = row['sympy_format']

  # get complexity of equation
  complexity = row['complexity'] #PySR automatically calculates this

  pred_func = sp.lambdify((t,x), proposed_equation, modules='numpy') #turns a SymPy equation into a Numpy function

  try:
    u_pred_values = pred_func(t_flat, x_flat)

    if np.isscalar(u_pred_values):
      u_pred_values = np.full_like(t_flat, u_pred_values)
  except:
    continue

  # INDENTATION FIXED: These now run normally if the try block succeeds
  # data loss
  data_loss = np.mean((Y-u_pred_values)**2)

  # Physics Residual and Physics loss
  du_dt = sp.diff(proposed_equation, t)
  d2u_dx2 = sp.diff(proposed_equation, x, 2)

  residual_expr = du_dt - (a * d2u_dx2)
  residual_func = sp.lambdify((t,x), residual_expr, modules='numpy')

  try:
    residual_values = residual_func(t_flat, x_flat)
    if np.isscalar(residual_values):
      residual_values = np.full_like(Y, residual_values)
    physics_loss = np.mean(residual_values**2)
  except:
    physics_loss = float('inf')

  # final score for an equation
  total_loss = data_loss + (weight*physics_loss)

  # saving results as a dictionary
  evaluated_equations.append({'equation': proposed_equation,'complexity': complexity, 'data_loss': data_loss, 'physics_loss': physics_loss, 'total_loss': total_loss})

evaluated_equations.sort(key=lambda item: item['total_loss'])

# get top 10 (or fewer)
top_equations = evaluated_equations[:10]

# results!
print("Top 10 Physics-Informed Equations")
print("-------------------------------------------------")
for rank, eq_data in enumerate(top_equations):
  print(f"Rank {rank + 1}:")
  print(f"Equation:   {eq_data['equation']}")
  print(f"Complexity: {eq_data['complexity']}")
  print(f"Total Loss: {eq_data['total_loss']:.6f}")
  print(f"Data Loss: {eq_data['data_loss']:.6f} | Physics Loss: {eq_data['physics_loss']:.6f}")
  print("-------------------------------------------------")

print(f"True Equation: {sp.exp(-a*t) * sp.sin(x)}")

Top 10 Physics-Informed Equations
-------------------------------------------------
Rank 1:
Equation:   exp(t*(-0.09965403))*sin(x*1.0009688)
Complexity: 9
Total Loss: 0.002503
Data Loss: 0.002502 | Physics Loss: 0.000000
-------------------------------------------------
Rank 2:
Equation:   exp(t*(-0.09981971))*sin(x)
Complexity: 7
Total Loss: 0.002504
Data Loss: 0.002504 | Physics Loss: 0.000000
-------------------------------------------------
Rank 3:
Equation:   (sin(x) - 1*(-0.0016893654))*0.786672
Complexity: 6
Total Loss: 0.039725
Data Loss: 0.009092 | Physics Loss: 0.003063
-------------------------------------------------
Rank 4:
Equation:   sin(x)*0.787573
Complexity: 4
Total Loss: 0.039796
Data Loss: 0.009093 | Physics Loss: 0.003070
-------------------------------------------------
Rank 5:
Equation:   sin(sin(x))
Complexity: 3
Total Loss: 0.058596
Data Loss: 0.014109 | Physics Loss: 0.004449
-------------------------------------------------
Rank 6:
Equation:   sin(x)
Complex

Thank you for checking out my PISR Heat Diffusion Toy Model!

I hope you liked it :)

In the future, I would like to explore adding an adaptive weight changing feature.